# Aria Mobile Integration Guide

**Complete guide for building mobile clients, cross-platform development, and mobile API integration.**

Learn how to build iOS, Android, and web-based mobile interfaces for Aria.


## Mobile Architecture

### Platform Support

```
Aria Backend APIs
├─ REST API (JSON)
├─ WebSocket (real-time)
├─ Server-Sent Events (SSE)
└─ GraphQL (optional)

Mobile Clients
├─ iOS (Swift, SwiftUI)
├─ Android (Kotlin, Jetpack Compose)
├─ React Native (cross-platform)
├─ Flutter (cross-platform)
└─ Progressive Web App (PWA)
```

### API Endpoints for Mobile

```
Base URL: https://api.aria.example.com/v1

Chat API
├─ POST /chat/completions       # Non-streaming completion
├─ GET /chat/stream             # SSE streaming
├─ POST /chat/memory            # Semantic memory
└─ GET /chat/history            # Conversation history

Aria Character API
├─ POST /aria/command           # Execute command
├─ GET /aria/state              # Current state
├─ POST /aria/world             # Generate world
└─ GET /aria/schema             # Available actions

User & Auth
├─ POST /auth/register          # Sign up
├─ POST /auth/login             # Login
├─ POST /auth/refresh           # Refresh token
└─ GET /auth/profile            # User profile

Subscriptions
├─ GET /subscriptions/plans     # Available plans
├─ POST /subscriptions/create   # Create subscription
└─ GET /subscriptions/status    # Current status
```

---

## iOS Development

### SwiftUI Chat Component

```swift
// ChatView.swift
import SwiftUI

struct ChatView: View {
    @StateObject private var viewModel = ChatViewModel()
    @State private var userInput = ""

    var body: some View {
        VStack(spacing: 0) {
            // Messages List
            ScrollView {
                VStack(alignment: .leading, spacing: 12) {
                    ForEach(viewModel.messages) { message in
                        ChatBubble(message: message)
                    }
                }
                .padding()
            }

            // Input Area
            HStack(spacing: 8) {
                TextField("Type a message...", text: $userInput)
                    .textFieldStyle(.roundedBorder)
                    .padding()

                Button(action: sendMessage) {
                    Image(systemName: "paperplane.fill")
                        .foregroundColor(.blue)
                }
                .padding()
                .disabled(userInput.isEmpty || viewModel.isLoading)
            }
            .background(Color(.systemGray6))
        }
        .onAppear {
            viewModel.connect()
        }
    }

    private func sendMessage() {
        viewModel.sendMessage(userInput)
        userInput = ""
    }
}

struct ChatBubble: View {
    let message: ChatMessage

    var body: some View {
        HStack(alignment: .bottom) {
            if message.isUser {
                Spacer()
                Text(message.content)
                    .padding(12)
                    .background(Color.blue)
                    .foregroundColor(.white)
                    .cornerRadius(16)
            } else {
                Text(message.content)
                    .padding(12)
                    .background(Color(.systemGray5))
                    .cornerRadius(16)
                Spacer()
            }
        }
    }
}

// ViewModel
class ChatViewModel: NSObject, ObservableObject, URLSessionWebSocketDelegate {
    @Published var messages: [ChatMessage] = []
    @Published var isLoading = false

    private var webSocket: URLWebSocketTask?
    private let apiKey: String

    override init() {
        self.apiKey = UserDefaults.standard.string(forKey: "api_key") ?? ""
    }

    func connect() {
        let url = URL(string: "wss://api.aria.example.com/v1/chat/stream")!
        var request = URLRequest(url: url)
        request.setValue("Bearer \(apiKey)", forHTTPHeaderField: "Authorization")

        let session = URLSession(configuration: .default, delegate: self, delegateQueue: nil)
        webSocket = session.webSocketTask(with: request)
        webSocket?.resume()
        receiveMessage()
    }

    func sendMessage(_ content: String) {
        isLoading = true
        let message = ChatMessage(id: UUID(), content: content, isUser: true, timestamp: Date())
        DispatchQueue.main.async {
            self.messages.append(message)
        }

        let data = ["message": content]
        if let jsonData = try? JSONEncoder().encode(data),
           let string = String(data: jsonData, encoding: .utf8) {
            webSocket?.send(.string(string)) { _ in }
        }
    }

    private func receiveMessage() {
        webSocket?.receive { [weak self] result in
            switch result {
            case .success(let message):
                switch message {
                case .string(let text):
                    DispatchQueue.main.async {
                        let aiMessage = ChatMessage(
                            id: UUID(),
                            content: text,
                            isUser: false,
                            timestamp: Date()
                        )
                        self?.messages.append(aiMessage)
                        self?.isLoading = false
                    }
                case .data(let data):
                    // Handle binary data
                    break
                @unknown default:
                    break
                }
                self?.receiveMessage()
            case .failure(let error):
                print("WebSocket error: \(error)")
            }
        }
    }
}

struct ChatMessage: Identifiable, Codable {
    let id: UUID
    let content: String
    let isUser: Bool
    let timestamp: Date
}
```

### Authentication Flow

```swift
// AuthManager.swift
import Foundation

class AuthManager: ObservableObject {
    @Published var isAuthenticated = false
    @Published var currentUser: User?

    private let baseURL = "https://api.aria.example.com/v1"

    func register(email: String, password: String) async {
        let payload = ["email": email, "password": password]

        guard let url = URL(string: "\(baseURL)/auth/register") else { return }
        var request = URLRequest(url: url)
        request.httpMethod = "POST"
        request.setValue("application/json", forHTTPHeaderField: "Content-Type")
        request.httpBody = try? JSONEncoder().encode(payload)

        do {
            let (data, _) = try await URLSession.shared.data(for: request)
            let response = try JSONDecoder().decode(AuthResponse.self, from: data)

            DispatchQueue.main.async {
                KeychainManager.saveToken(response.accessToken)
                self.currentUser = response.user
                self.isAuthenticated = true
            }
        } catch {
            print("Registration failed: \(error)")
        }
    }

    func logout() {
        KeychainManager.deleteToken()
        isAuthenticated = false
        currentUser = nil
    }
}

// Keychain Manager
struct KeychainManager {
    static func saveToken(_ token: String) {
        let query: [String: Any] = [
            kSecClass as String: kSecClassGenericPassword,
            kSecAttrAccount as String: "aria_auth_token",
            kSecValueData as String: token.data(using: .utf8)!
        ]
        SecItemDelete(query as CFDictionary)
        SecItemAdd(query as CFDictionary, nil)
    }

    static func getToken() -> String? {
        let query: [String: Any] = [
            kSecClass as String: kSecClassGenericPassword,
            kSecAttrAccount as String: "aria_auth_token",
            kSecReturnData as String: true
        ]

        var result: AnyObject?
        SecItemCopyMatching(query as CFDictionary, &result)

        if let data = result as? Data {
            return String(data: data, encoding: .utf8)
        }
        return nil
    }

    static func deleteToken() {
        let query: [String: Any] = [
            kSecClass as String: kSecClassGenericPassword,
            kSecAttrAccount as String: "aria_auth_token"
        ]
        SecItemDelete(query as CFDictionary)
    }
}
```

---

## Android Development

### Kotlin Compose Chat UI

```kotlin
// ChatScreen.kt
@Composable
fun ChatScreen(viewModel: ChatViewModel = viewModel()) {
    val messages by viewModel.messages.collectAsState()
    val isLoading by viewModel.isLoading.collectAsState()
    var userInput by remember { mutableStateOf("") }

    Column(modifier = Modifier.fillMaxSize()) {
        // Messages
        LazyColumn(
            modifier = Modifier
                .weight(1f)
                .fillMaxWidth()
                .padding(8.dp),
            reverseLayout = true
        ) {
            items(messages.reversed()) { message ->
                ChatMessageItem(message = message)
            }
        }

        // Input
        Row(
            modifier = Modifier
                .fillMaxWidth()
                .padding(8.dp),
            horizontalArrangement = Arrangement.spacedBy(8.dp)
        ) {
            TextField(
                value = userInput,
                onValueChange = { userInput = it },
                placeholder = { Text("Type a message...") },
                modifier = Modifier
                    .weight(1f)
                    .align(Alignment.CenterVertically)
            )

            Button(
                onClick = {
                    viewModel.sendMessage(userInput)
                    userInput = ""
                },
                enabled = userInput.isNotEmpty() && !isLoading
            ) {
                Icon(Icons.Default.Send, contentDescription = "Send")
            }
        }
    }
}

@Composable
fun ChatMessageItem(message: ChatMessage) {
    Row(
        modifier = Modifier
            .fillMaxWidth()
            .padding(8.dp),
        horizontalArrangement = if (message.isUser)
            Arrangement.End else Arrangement.Start
    ) {
        Surface(
            shape = RoundedCornerShape(12.dp),
            color = if (message.isUser) Color.Blue else Color.LightGray,
            modifier = Modifier.widthIn(max = 300.dp)
        ) {
            Text(
                text = message.content,
                modifier = Modifier.padding(12.dp),
                color = if (message.isUser) Color.White else Color.Black
            )
        }
    }
}

// ViewModel
@HiltViewModel
class ChatViewModel @Inject constructor(
    private val chatRepository: ChatRepository
) : ViewModel() {

    private val _messages = MutableStateFlow<List<ChatMessage>>(emptyList())
    val messages: StateFlow<List<ChatMessage>> = _messages.asStateFlow()

    private val _isLoading = MutableStateFlow(false)
    val isLoading: StateFlow<Boolean> = _isLoading.asStateFlow()

    private var webSocket: WebSocket? = null

    fun sendMessage(content: String) {
        viewModelScope.launch {
            _isLoading.value = true

            val userMessage = ChatMessage(
                id = UUID.randomUUID().toString(),
                content = content,
                isUser = true,
                timestamp = System.currentTimeMillis()
            )
            _messages.value = _messages.value + userMessage

            try {
                val response = chatRepository.sendMessage(content)
                val aiMessage = ChatMessage(
                    id = UUID.randomUUID().toString(),
                    content = response,
                    isUser = false,
                    timestamp = System.currentTimeMillis()
                )
                _messages.value = _messages.value + aiMessage
            } catch (e: Exception) {
                // Handle error
            } finally {
                _isLoading.value = false
            }
        }
    }
}
```

---

## React Native Cross-Platform

### Chat Component

```typescript
// ChatScreen.tsx
import React, { useState, useEffect } from 'react';
import {
  View,
  Text,
  TextInput,
  TouchableOpacity,
  FlatList,
  StyleSheet,
  ActivityIndicator,
} from 'react-native';
import { useChat } from './hooks/useChat';

interface Message {
  id: string;
  content: string;
  isUser: boolean;
  timestamp: number;
}

export const ChatScreen: React.FC = () => {
  const { messages, loading, sendMessage } = useChat();
  const [input, setInput] = useState('');

  const handleSend = () => {
    if (input.trim()) {
      sendMessage(input);
      setInput('');
    }
  };

  const renderMessage = ({ item }: { item: Message }) => (
    <View
      style={[
        styles.messageBubble,
        item.isUser ? styles.userMessage : styles.aiMessage,
      ]}
    >
      <Text
        style={[
          styles.messageText,
          item.isUser ? styles.userText : styles.aiText,
        ]}
      >
        {item.content}
      </Text>
    </View>
  );

  return (
    <View style={styles.container}>
      <FlatList
        data={messages}
        renderItem={renderMessage}
        keyExtractor={(item) => item.id}
        inverted
        contentContainerStyle={styles.messageList}
      />

      <View style={styles.inputContainer}>
        <TextInput
          style={styles.input}
          placeholder="Type a message..."
          value={input}
          onChangeText={setInput}
          editable={!loading}
        />
        <TouchableOpacity
          style={[styles.sendButton, loading && styles.sendButtonDisabled]}
          onPress={handleSend}
          disabled={loading || !input.trim()}
        >
          {loading ? (
            <ActivityIndicator color="white" />
          ) : (
            <Text style={styles.sendButtonText}>Send</Text>
          )}
        </TouchableOpacity>
      </View>
    </View>
  );
};

const styles = StyleSheet.create({
  container: {
    flex: 1,
    backgroundColor: '#fff',
  },
  messageList: {
    padding: 10,
  },
  messageBubble: {
    marginVertical: 5,
    borderRadius: 12,
    padding: 10,
    maxWidth: '80%',
  },
  userMessage: {
    alignSelf: 'flex-end',
    backgroundColor: '#007AFF',
  },
  aiMessage: {
    alignSelf: 'flex-start',
    backgroundColor: '#E5E5EA',
  },
  messageText: {
    fontSize: 16,
  },
  userText: {
    color: 'white',
  },
  aiText: {
    color: 'black',
  },
  inputContainer: {
    flexDirection: 'row',
    padding: 10,
    borderTopWidth: 1,
    borderTopColor: '#E5E5EA',
  },
  input: {
    flex: 1,
    borderWidth: 1,
    borderColor: '#ddd',
    borderRadius: 20,
    paddingHorizontal: 15,
    paddingVertical: 10,
  },
  sendButton: {
    marginLeft: 10,
    backgroundColor: '#007AFF',
    borderRadius: 20,
    paddingHorizontal: 20,
    justifyContent: 'center',
  },
  sendButtonDisabled: {
    opacity: 0.5,
  },
  sendButtonText: {
    color: 'white',
    fontWeight: 'bold',
  },
});
```

---

## Progressive Web App (PWA)

### Service Worker

```typescript
// service-worker.ts
const CACHE_VERSION = "aria-v1"
const ASSETS_TO_CACHE = [
    "/",
    "/index.html",
    "/styles.css",
    "/app.js",
    "/manifest.json",
]

// Install event
self.addEventListener("install", (event: ExtendableEvent) => {
    event.waitUntil(
        (async () => {
            const cache = await caches.open(CACHE_VERSION)
            await cache.addAll(ASSETS_TO_CACHE)
            await self.skipWaiting()
        })(),
    )
})

// Activate event
self.addEventListener("activate", (event: ExtendableEvent) => {
    event.waitUntil(
        (async () => {
            const cacheNames = await caches.keys()
            const toDelete = cacheNames.filter(name => name !== CACHE_VERSION)
            await Promise.all(toDelete.map(name => caches.delete(name)))
            await self.clients.claim()
        })(),
    )
})

// Fetch event - Network first, then cache
self.addEventListener("fetch", (event: FetchEvent) => {
    const { request } = event

    if (request.method !== "GET") {
        return
    }

    event.respondWith(
        (async () => {
            try {
                const response = await fetch(request)
                const cache = await caches.open(CACHE_VERSION)
                cache.put(request, response.clone())
                return response
            } catch {
                const cached = await caches.match(request)
                return cached || new Response("Offline", { status: 503 })
            }
        })(),
    )
})
```

### PWA Manifest

```json
{
    "name": "Aria Mobile Chat",
    "short_name": "Aria Chat",
    "description": "AI chat and character interaction",
    "start_url": "/",
    "display": "standalone",
    "orientation": "portrait",
    "background_color": "#ffffff",
    "theme_color": "#007AFF",
    "icons": [
        {
            "src": "/icon-192.png",
            "sizes": "192x192",
            "type": "image/png",
            "purpose": "any"
        },
        {
            "src": "/icon-512.png",
            "sizes": "512x512",
            "type": "image/png",
            "purpose": "maskable"
        }
    ]
}
```


## API Best Practices for Mobile

### Rate Limiting & Throttling

```python
# function_app.py
from functools import wraps
from datetime import datetime, timedelta

class RateLimiter:
    def __init__(self, max_requests=100, window_seconds=60):
        self.max_requests = max_requests
        self.window = window_seconds
        self.requests = {}

    def is_allowed(self, client_id: str) -> bool:
        now = datetime.now()
        if client_id not in self.requests:
            self.requests[client_id] = []

        # Remove old requests outside window
        self.requests[client_id] = [
            req_time for req_time in self.requests[client_id]
            if (now - req_time).total_seconds() < self.window
        ]

        if len(self.requests[client_id]) >= self.max_requests:
            return False

        self.requests[client_id].append(now)
        return True

# Usage in endpoint
rate_limiter = RateLimiter(max_requests=100, window_seconds=60)

@app.route("/api/chat/completions", methods=["POST"])
def chat_completion(req: func.HttpRequest):
    client_id = req.headers.get("X-Client-ID", "unknown")

    if not rate_limiter.is_allowed(client_id):
        return func.HttpResponse(
            json.dumps({"error": "Rate limit exceeded"}),
            status_code=429,
            headers={"Retry-After": "60"}
        )

    # Process request...
```

### Offline Sync

```typescript
// offlineSync.ts
interface PendingAction {
    id: string
    action: "chat" | "command"
    data: any
    timestamp: number
    retries: number
}

class OfflineSyncManager {
    private db: IDBDatabase
    private maxRetries = 3

    async savePendingAction(action: PendingAction): Promise<void> {
        const tx = this.db.transaction("pending_actions", "readwrite")
        await tx.objectStore("pending_actions").add(action)
    }

    async syncPendingActions(): Promise<void> {
        const tx = this.db.transaction("pending_actions", "readonly")
        const actions = await tx.objectStore("pending_actions").getAll()

        for (const action of actions) {
            try {
                await this.executeAction(action)
                await this.removePendingAction(action.id)
            } catch (error) {
                if (action.retries < this.maxRetries) {
                    action.retries++
                    await this.savePendingAction(action)
                } else {
                    await this.removePendingAction(action.id)
                }
            }
        }
    }

    private async executeAction(action: PendingAction): Promise<void> {
        const response = await fetch("/api/sync/action", {
            method: "POST",
            body: JSON.stringify(action.data),
        })
        if (!response.ok) throw new Error("Sync failed")
    }

    private async removePendingAction(id: string): Promise<void> {
        const tx = this.db.transaction("pending_actions", "readwrite")
        await tx.objectStore("pending_actions").delete(id)
    }
}
```

### Mobile-Specific Response Optimization

```python
# Compress responses for mobile
@app.route("/api/chat/completions", methods=["POST"])
def chat_completion_mobile(req: func.HttpRequest):
    accept_encoding = req.headers.get("Accept-Encoding", "")

    response_data = {
        "message": "Hello from Aria",
        "tokens": 42
    }

    if "gzip" in accept_encoding:
        import gzip
        compressed = gzip.compress(json.dumps(response_data).encode())
        return func.HttpResponse(
            compressed,
            status_code=200,
            headers={
                "Content-Encoding": "gzip",
                "Content-Type": "application/json"
            }
        )

    return func.HttpResponse(json.dumps(response_data), status_code=200)
```

---

## Mobile Testing

### iOS UI Testing

```swift
// ChatViewTests.swift
import XCTest

class ChatViewTests: XCTestCase {
    var app: XCUIApplication!

    override func setUp() {
        super.setUp()
        app = XCUIApplication()
        app.launch()
    }

    func testSendMessage() {
        let textField = app.textFields["messageInput"]
        textField.tap()
        textField.typeText("Hello Aria")

        app.buttons["send"].tap()

        let message = app.staticTexts["Hello Aria"]
        XCTAssertTrue(message.exists)
    }

    func testLoadingState() {
        let textField = app.textFields["messageInput"]
        textField.tap()
        textField.typeText("Test")
        app.buttons["send"].tap()

        let spinner = app.activityIndicators["loading"]
        XCTAssertTrue(spinner.exists)
    }
}
```

### Android UI Testing

```kotlin
// ChatScreenTest.kt
@RunWith(AndroidJUnit4::class)
class ChatScreenTest {
    @get:Rule
    val composeTestRule = createComposeRule()

    @Test
    fun testSendMessage() {
        composeTestRule.setContent {
            ChatScreen()
        }

        composeTestRule
            .onNodeWithTag("messageInput")
            .performTextInput("Hello Aria")

        composeTestRule
            .onNodeWithTag("sendButton")
            .performClick()

        composeTestRule
            .onNodeWithText("Hello Aria")
            .assertExists()
    }
}
```

---

## Mobile Best Practices

- [ ] Responsive design (4-6 inch screens minimum)
- [ ] Touch-friendly UI (min 44x44 pt buttons)
- [ ] Offline support with sync
- [ ] Optimize for mobile bandwidth
- [ ] Battery-aware API calls
- [ ] Secure token storage (Keychain/Keystore)
- [ ] HTTPS everywhere
- [ ] Rate limiting on mobile
- [ ] Implement retry logic
- [ ] Native platform patterns (not web UI)
- [ ] Test on real devices
- [ ] Monitor crash rates
